# Notebook 01 — Setup & Data Load
## JPMC Consumer Banking Performance Engineering HOL

**Objective**: Stand up the environment and load 500M+ synthetic banking transactions in TWO ways:
1. **TRANSACTIONS** — loaded chronologically (naturally ordered)
2. **TRANSACTIONS_UNORDERED** — same data, randomly shuffled (no natural order)

**Time**: ~8 minutes (data generation runs on an XL warehouse)

---
## Step 1: Create Environment

In [ ]:
-- Environment setup
USE ROLE SYSADMIN;

CREATE OR REPLACE DATABASE JPMC_PERF_HOL;
CREATE SCHEMA JPMC_PERF_HOL.CONSUMER_BANKING;

-- Warehouses for testing different sizes
CREATE WAREHOUSE IF NOT EXISTS HOL_XS  WAREHOUSE_SIZE = 'XSMALL'  AUTO_SUSPEND = 60 AUTO_RESUME = TRUE;
CREATE WAREHOUSE IF NOT EXISTS HOL_M   WAREHOUSE_SIZE = 'MEDIUM'  AUTO_SUSPEND = 60 AUTO_RESUME = TRUE;
CREATE WAREHOUSE IF NOT EXISTS HOL_L   WAREHOUSE_SIZE = 'LARGE'   AUTO_SUSPEND = 60 AUTO_RESUME = TRUE;
CREATE WAREHOUSE IF NOT EXISTS HOL_XL  WAREHOUSE_SIZE = 'XLARGE'  AUTO_SUSPEND = 60 AUTO_RESUME = TRUE;

-- Use XL for data loading (500M+ rows)
USE WAREHOUSE HOL_XL;
USE SCHEMA JPMC_PERF_HOL.CONSUMER_BANKING;

---
## Step 2: Create Tables

In [ ]:
CREATE OR REPLACE TABLE CUSTOMERS (
    customer_id       VARCHAR(20)   PRIMARY KEY,
    first_name        VARCHAR(50),
    last_name         VARCHAR(50),
    ssn_hash          VARCHAR(64),
    date_of_birth     DATE,
    email             VARCHAR(100),
    phone             VARCHAR(20),
    address_state     VARCHAR(2),
    address_zip       VARCHAR(10),
    segment           VARCHAR(20),
    onboarding_date   DATE,
    risk_score        NUMBER(5,2)
);

CREATE OR REPLACE TABLE ACCOUNTS (
    account_id        VARCHAR(20)   PRIMARY KEY,
    customer_id       VARCHAR(20),
    account_type      VARCHAR(20),
    open_date         DATE,
    close_date        DATE,
    current_balance   NUMBER(15,2),
    credit_limit      NUMBER(15,2),
    branch_id         VARCHAR(10),
    status            VARCHAR(10)
);

CREATE OR REPLACE TABLE TRANSACTIONS (
    transaction_id          VARCHAR(30)   PRIMARY KEY,
    account_id              VARCHAR(20),
    transaction_date        TIMESTAMP_NTZ,
    post_date               DATE,
    merchant_name           VARCHAR(100),
    merchant_category_code  VARCHAR(10),
    transaction_type        VARCHAR(20),
    amount                  NUMBER(12,2),
    currency_code           VARCHAR(3),
    channel                 VARCHAR(10),
    is_international        BOOLEAN,
    fraud_flag              BOOLEAN,
    fraud_score             NUMBER(5,4)
);

CREATE OR REPLACE TABLE FRAUD_ALERTS (
    alert_id              VARCHAR(30)   PRIMARY KEY,
    transaction_id        VARCHAR(30),
    alert_timestamp       TIMESTAMP_NTZ,
    alert_type            VARCHAR(20),
    risk_level            VARCHAR(10),
    resolution            VARCHAR(20),
    resolved_by           VARCHAR(50),
    resolution_timestamp  TIMESTAMP_NTZ
);

CREATE OR REPLACE TABLE BRANCHES (
    branch_id     VARCHAR(10)  PRIMARY KEY,
    branch_name   VARCHAR(100),
    region        VARCHAR(20),
    state         VARCHAR(2),
    city          VARCHAR(50),
    manager_name  VARCHAR(100)
);

---
## Step 3: Generate Synthetic Data (500M Transactions)

Transactions are loaded in **chronological order** by `transaction_date`.  
This is intentional — it creates natural clustering we'll explore in Notebook 02.

In [ ]:
-- 5,000 branches
INSERT INTO BRANCHES
SELECT
    'BR-' || LPAD(SEQ4(), 5, '0'),
    'Branch ' || SEQ4(),
    CASE MOD(SEQ4(), 5)
        WHEN 0 THEN 'Northeast' WHEN 1 THEN 'Southeast'
        WHEN 2 THEN 'Midwest'   WHEN 3 THEN 'West' ELSE 'Southwest' END,
    CASE MOD(SEQ4(), 10)
        WHEN 0 THEN 'NY' WHEN 1 THEN 'NJ' WHEN 2 THEN 'CT' WHEN 3 THEN 'FL'
        WHEN 4 THEN 'TX' WHEN 5 THEN 'CA' WHEN 6 THEN 'IL' WHEN 7 THEN 'OH'
        WHEN 8 THEN 'PA' ELSE 'GA' END,
    'City-' || MOD(SEQ4(), 200),
    'Manager-' || SEQ4()
FROM TABLE(GENERATOR(ROWCOUNT => 5000));

In [ ]:
-- 10M customers
INSERT INTO CUSTOMERS
SELECT
    'CUST-' || LPAD(SEQ8(), 10, '0'),
    'First' || MOD(SEQ8(), 5000),
    'Last' || MOD(SEQ8(), 8000),
    SHA2(SEQ8()::VARCHAR || 'ssn', 256),
    DATEADD(day, -UNIFORM(7000, 25000, RANDOM()), CURRENT_DATE()),
    'user' || SEQ8() || '@email.com',
    '555-' || LPAD(MOD(SEQ8(), 10000000)::VARCHAR, 7, '0'),
    CASE MOD(SEQ8(), 10)
        WHEN 0 THEN 'NY' WHEN 1 THEN 'NJ' WHEN 2 THEN 'CT' WHEN 3 THEN 'FL'
        WHEN 4 THEN 'TX' WHEN 5 THEN 'CA' WHEN 6 THEN 'IL' WHEN 7 THEN 'OH'
        WHEN 8 THEN 'PA' ELSE 'GA' END,
    LPAD(MOD(SEQ8(), 99999)::VARCHAR, 5, '0'),
    CASE WHEN MOD(SEQ8(), 100) < 70 THEN 'mass_market'
         WHEN MOD(SEQ8(), 100) < 95 THEN 'affluent'
         ELSE 'private_banking' END,
    DATEADD(day, -UNIFORM(100, 3650, RANDOM()), CURRENT_DATE()),
    ROUND(UNIFORM(1, 100, RANDOM()) / 10.0, 2)
FROM TABLE(GENERATOR(ROWCOUNT => 10000000));

In [ ]:
-- 25M accounts
INSERT INTO ACCOUNTS
SELECT
    'ACC-' || LPAD(SEQ8(), 10, '0'),
    'CUST-' || LPAD(MOD(SEQ8(), 10000000), 10, '0'),
    CASE MOD(SEQ8(), 4)
        WHEN 0 THEN 'checking' WHEN 1 THEN 'savings'
        WHEN 2 THEN 'credit_card' ELSE 'mortgage' END,
    DATEADD(day, -UNIFORM(100, 3650, RANDOM()), CURRENT_DATE()),
    CASE WHEN MOD(SEQ8(), 20) = 0 THEN DATEADD(day, -UNIFORM(1, 100, RANDOM()), CURRENT_DATE()) ELSE NULL END,
    ROUND(UNIFORM(-5000, 250000, RANDOM())::NUMBER(15,2) / 100, 2),
    CASE WHEN MOD(SEQ8(), 4) = 2 THEN UNIFORM(5000, 50000, RANDOM()) ELSE NULL END,
    'BR-' || LPAD(MOD(SEQ8(), 5000)::VARCHAR, 5, '0'),
    CASE WHEN MOD(SEQ8(), 20) = 0 THEN 'closed'
         WHEN MOD(SEQ8(), 50) = 0 THEN 'frozen' ELSE 'active' END
FROM TABLE(GENERATOR(ROWCOUNT => 25000000));

In [ ]:
-- 500M transactions (chronologically ordered — creates natural clustering on transaction_date)
INSERT INTO TRANSACTIONS
SELECT
    'TXN-' || LPAD(SEQ8(), 12, '0'),
    'ACC-' || LPAD(UNIFORM(0, 24999999, RANDOM())::INT, 10, '0'),
    DATEADD(second, (SEQ8() * 0.19)::INT, '2022-01-01'::TIMESTAMP_NTZ),
    DATEADD(day, 1, DATEADD(second, (SEQ8() * 0.19)::INT, '2022-01-01'::TIMESTAMP_NTZ))::DATE,
    CASE MOD(SEQ8(), 20)
        WHEN 0 THEN 'AMAZON' WHEN 1 THEN 'WALMART' WHEN 2 THEN 'TARGET'
        WHEN 3 THEN 'STARBUCKS' WHEN 4 THEN 'SHELL GAS' WHEN 5 THEN 'UBER'
        WHEN 6 THEN 'NETFLIX' WHEN 7 THEN 'COSTCO' WHEN 8 THEN 'HOME DEPOT'
        WHEN 9 THEN 'CHASE PAYMENT' WHEN 10 THEN 'VENMO TRANSFER'
        WHEN 11 THEN 'ATM WITHDRAWAL' WHEN 12 THEN 'APPLE' WHEN 13 THEN 'WHOLE FOODS'
        WHEN 14 THEN 'CVS PHARMACY' WHEN 15 THEN 'WESTERN UNION'
        WHEN 16 THEN 'CRYPTO EXCHANGE' WHEN 17 THEN 'AIRLINE TICKETS'
        WHEN 18 THEN 'HOTEL BOOKING' ELSE 'MISC RETAIL' END,
    LPAD(MOD(SEQ8(), 100)::VARCHAR, 4, '0'),
    CASE MOD(SEQ8(), 6)
        WHEN 0 THEN 'purchase' WHEN 1 THEN 'payment' WHEN 2 THEN 'transfer'
        WHEN 3 THEN 'ATM' WHEN 4 THEN 'fee' ELSE 'refund' END,
    ROUND(ABS(UNIFORM(1, 50000, RANDOM())::NUMBER(12,2)) / 100, 2),
    CASE WHEN MOD(SEQ8(), 50) = 0 THEN 'GBP'
         WHEN MOD(SEQ8(), 80) = 0 THEN 'EUR' ELSE 'USD' END,
    CASE MOD(SEQ8(), 5)
        WHEN 0 THEN 'branch' WHEN 1 THEN 'online' WHEN 2 THEN 'mobile'
        WHEN 3 THEN 'ATM' ELSE 'POS' END,
    MOD(SEQ8(), 50) = 0,
    MOD(SEQ8(), 200) = 0,
    ROUND(UNIFORM(0, 10000, RANDOM()) / 10000.0, 4)
FROM TABLE(GENERATOR(ROWCOUNT => 500000000));

---
## Step 4: Create TRANSACTIONS_UNORDERED (Same Data, Randomly Shuffled)

This table has **identical data** but loaded in **random order** — destroying natural clustering.  
In Notebook 02 we'll run the same query on both tables to see the performance impact.

In [ ]:
-- Same 500M rows but randomly shuffled — NO natural ordering on any column
CREATE OR REPLACE TABLE TRANSACTIONS_UNORDERED AS
SELECT *
FROM TRANSACTIONS
ORDER BY RANDOM();

In [ ]:
-- 5M fraud alerts
INSERT INTO FRAUD_ALERTS
SELECT
    'ALT-' || LPAD(SEQ8(), 10, '0'),
    'TXN-' || LPAD(MOD(SEQ8(), 500000000), 12, '0'),
    DATEADD(second, UNIFORM(60, 3600, RANDOM()), DATEADD(second, SEQ8() * 19, '2022-01-01'::TIMESTAMP_NTZ)),
    CASE MOD(SEQ8(), 4)
        WHEN 0 THEN 'velocity' WHEN 1 THEN 'geo_anomaly'
        WHEN 2 THEN 'amount_anomaly' ELSE 'device_new' END,
    CASE MOD(SEQ8(), 4)
        WHEN 0 THEN 'low' WHEN 1 THEN 'medium'
        WHEN 2 THEN 'high' ELSE 'critical' END,
    CASE WHEN MOD(SEQ8(), 3) = 0 THEN 'confirmed_fraud'
         WHEN MOD(SEQ8(), 3) = 1 THEN 'false_positive' ELSE 'pending' END,
    'Analyst-' || MOD(SEQ8(), 200),
    CASE WHEN MOD(SEQ8(), 3) != 2
         THEN DATEADD(minute, UNIFORM(5, 1440, RANDOM()), DATEADD(second, SEQ8() * 19, '2022-01-01'::TIMESTAMP_NTZ))
         ELSE NULL END
FROM TABLE(GENERATOR(ROWCOUNT => 5000000));

---
## Step 5: Verify Data

In [ ]:
SELECT 'CUSTOMERS' AS table_name, COUNT(*) AS row_count FROM CUSTOMERS
UNION ALL SELECT 'ACCOUNTS', COUNT(*) FROM ACCOUNTS
UNION ALL SELECT 'TRANSACTIONS (ordered)', COUNT(*) FROM TRANSACTIONS
UNION ALL SELECT 'TRANSACTIONS_UNORDERED (shuffled)', COUNT(*) FROM TRANSACTIONS_UNORDERED
UNION ALL SELECT 'FRAUD_ALERTS', COUNT(*) FROM FRAUD_ALERTS
UNION ALL SELECT 'BRANCHES', COUNT(*) FROM BRANCHES
ORDER BY row_count DESC;

---
## Setup Complete

| Table | Rows | Key Point |
|-------|------|---------|
| TRANSACTIONS | 500M | Loaded chronologically — **naturally ordered** on `transaction_date` |
| TRANSACTIONS_UNORDERED | 500M | Same data, randomly shuffled — **no natural order** |
| ACCOUNTS | 25M | Join target for customer-level queries |
| CUSTOMERS | 10M | Demographics and segmentation |
| FRAUD_ALERTS | 5M | Fraud investigation scenarios |
| BRANCHES | 5K | Reference/dimension table |

**Next** → [Notebook 02 — Micro-Partitions & Natural Data Ordering](./Notebook_02_Natural_Ordering.ipynb)